In [1]:
!pip install fastapi uvicorn keras-facenet mtcnn tensorflow pillow python-multipart "uvicorn[standard]" requests

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.7/517.7 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.8/456.8 kB 32.3 MB/s eta 0:00:00
  Created wheel for keras-facenet: filename=keras_facenet-0.3.2-py3-none-any.whl size=10367 sha256=1ec363ce39d943a81b4c7adc65f4b7bf03035c4c90c148926b423b0e9dcffd2b
  Stored in directory: /root/.cache/pip/wheels/05/b0/f5/19ac49fedc10b1df3ee56b096edbcfa39d45794fccc6bcdbbf
Successfully built keras-facenet


In [2]:
%%writefile train_facenet.py
"""
Training Facenet model
"""
from keras_facenet import FaceNet

def prepare_model():
    print("Loading pre-trained FaceNet model...")
    embedder = FaceNet()
    print("Model is ready to use for embeddings.")
    return embedder

if __name__ == "__main__":
    _ = prepare_model()
    print("Training/Preparation step completed successfully.")

Writing train_facenet.py


In [3]:
!python train_facenet.py

2025-11-15 18:16:56.377294: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763230616.404424    1606 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763230616.409418    1606 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763230616.420679    1606 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1763230616.420711    1606 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1763230616.420715    1606 computation_placer.cc:177] computation placer alr

In [4]:
%%writefile face_verification.py
"""
Testing/Inference file for FaceNet
"""
import numpy as np
from PIL import Image
from keras_facenet import FaceNet
from mtcnn.mtcnn import MTCNN

_embedder = None
_detector = None

def load_model():
    global _embedder, _detector
    if _embedder is None:
        _embedder = FaceNet()
    if _detector is None:
        _detector = MTCNN()
    return _embedder, _detector

def _extract_face(image: Image.Image, detector, required_size=(160, 160)):
    pixels = np.asarray(image.convert("RGB"))
    results = detector.detect_faces(pixels)
    if len(results) == 0:
        return None, []

    # Choose largest face
    results = sorted(results, key=lambda r: r["box"][2] * r["box"][3], reverse=True)

    face = results[0]
    x, y, w, h = face["box"]
    x, y = max(0, x), max(0, y)
    cropped = pixels[y:y+h, x:x+w]
    face_image = Image.fromarray(cropped).resize(required_size)

    bounding_box = [int(x), int(y), int(w), int(h)]
    return face_image, [bounding_box]

def _get_embedding(face_image: Image.Image, embedder):
    """
    Converts a cropped face image to a 512-d embedding using FaceNet.
    """
    face_array = np.asarray(face_image).astype("float32")
    mean, std = face_array.mean(), face_array.std()
    face_array = (face_array - mean) / (std + 1e-6)
    samples = np.expand_dims(face_array, axis=0)  # (1,160,160,3)
    embedding = embedder.embeddings(samples)
    return embedding[0]

def _cosine_similarity(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-6))

def verify_faces(image1: Image.Image, image2: Image.Image, threshold: float = 0.5):
    """
    Given two People's Images,it returns:
      verification_result: "same person" / "different person" / "face_not_detected"
    """
    embedder, detector = load_model()

    face1, boxes1 = _extract_face(image1, detector)
    face2, boxes2 = _extract_face(image2, detector)

    if face1 is None or face2 is None:
        return {
            "verification_result": "face_not_detected",
            "similarity_score": None,
            "bounding_boxes": {
                "image1": boxes1,
                "image2": boxes2
            }
        }

    emb1 = _get_embedding(face1, embedder)
    emb2 = _get_embedding(face2, embedder)

    sim = _cosine_similarity(emb1, emb2)
    result = "same person" if sim >= threshold else "different person"

    return {
        "verification_result": result,
        "similarity_score": sim,
        "bounding_boxes": {
            "image1": boxes1,
            "image2": boxes2
        }
    }

Writing face_verification.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [5]:
%%writefile main.py
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
from PIL import Image
import io

from face_verification import verify_faces

app = FastAPI(title="Face Authentication Service with FaceNet")

@app.post("/verify-faces")
async def verify_faces_endpoint(
    file1: UploadFile = File(...),
    file2: UploadFile = File(...)
):

    img_bytes1 = await file1.read()
    img_bytes2 = await file2.read()

    image1 = Image.open(io.BytesIO(img_bytes1))
    image2 = Image.open(io.BytesIO(img_bytes2))

    result = verify_faces(image1, image2)

    return JSONResponse(content=result)

Writing main.py


In [6]:
import uvicorn
import threading

def run_app():
    uvicorn.run("main:app", host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_app, daemon=True)
thread.start()
print("FastAPI server started on http://0.0.0.0:8000")

FastAPI server started on http://0.0.0.0:8000


In [ ]:
!ls '/content/drive/My Drive/'

In [7]:
# Download sample face images for testing
import requests
import os

# Download two sample faces
url1 = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg"
url2 = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg"

with open("face1.jpg", "wb") as f:
    f.write(requests.get(url1).content)

with open("face2.jpg", "wb") as f:
    f.write(requests.get(url2).content)

print("Sample images downloaded: face1.jpg, face2.jpg")

Sample images downloaded: face1.jpg, face2.jpg


In [8]:
# Testing the FastAPI endpoint
import requests
import time


time.sleep(2)

url = "http://127.0.0.1:8000/verify-faces"

files_req = {
    "file1": open("face1.jpg", "rb"),
    "file2": open("face2.jpg", "rb"),
}

print("Testing API endpoint...")
response = requests.post(url, files=files_req)

print("\nStatus code:", response.status_code)
print("\nResponse JSON:")
import json
print(json.dumps(response.json(), indent=2))

Testing API endpoint...
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
INFO:     127.0.0.1:45676 - "POST /verify-faces HTTP/1.1" 200 OK

Status code: 200

Response JSON:
{
  "verification_result": "same person",
  "similarity_score": 0.9999990463256836,
  "bounding_boxes": {
    "image1": [
      [
        210,
        187,
        142,
        206
      ]
    ],
    "image2": [
      [
        210,
        187,
        142,
        206
      ]
    ]
  }
}


In [ ]:
import pandas as pd

# Replace 'path/to/your/my_data.csv' with the actual path to your file in Google Drive
# Example: If your file is directly in 'My Drive', the path would be '/content/drive/My Drive/my_data.csv'
file_path = '/content/drive/My Drive/my_data.csv'

try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded {file_path}:")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")